# 01 · Load the data

**Project:** _title_ · **Team:** _names_

Downloads the raw data once, checks it, and caches it as Parquet in `data/cache/` (git-ignored). Re-run it only when you want fresh data, and say so in the README.

In [4]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv

DATA = Path("../data")
CACHE = DATA / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
load_dotenv(dotenv_path=find_dotenv())                                   # tokens live in ../.env, never in the notebook

True

## Metadata

Name, publisher, licence and last update of each dataset (see `03_open_data_apis` in the course repository).

In [3]:
DATASET = "mrus-ixpq"   # Socrata id of your dataset https://analisi.transparenciacatalunya.cat/api/v3/views/mrus-ixpq/query.json
meta = requests.get(f"https://analisi.transparenciacatalunya.cat/api/views/{DATASET}.json", timeout=30).json()
meta["name"], meta.get("attribution"), meta.get("license")

('Publicacions de recerca de Catalunya',
 'Departament de Recerca i Universitats',
 {'name': 'See Terms of Use'})

In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from sodapy import Socrata
from tqdm import tqdm

MAX_WORKERS = 4
MAX_RETRIES = 3
RETRY_BACKOFF = 5  # segundos base (5, 10, 20...)

def fetch_papers_from_year(client, year):
    """Descarga y cachea los papers de un año (síncrono)."""
    data = client.get_all(
        DATASET,
        select='date, doi, ruct, orcidResearchers, orcidDirectors, orcidUncheckedAuthors, orcidUncheckedDirectors',
        where=f'''doi IS NOT NULL 
            AND issn IS NOT NULL 
            AND orcidResearchers IS NOT NULL 
            AND ruct IS NOT NULL 
            AND date BETWEEN "{year}-01-01T00:00:00" AND "{year+1}-01-01T00:00:00" ''',
    )
    df = pd.DataFrame(data)
    if len(df) > 0:
        df = df.sort_values(by='date', ignore_index=True)
        df.to_parquet(path=f'../data/cache/papers_{year}.parquet')
    return year, len(df)


def fetch_with_retry(client, year):
    """Reintenta con backoff exponencial ante 429 / errores transitorios."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fetch_papers_from_year(client, year)
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(RETRY_BACKOFF * (2 ** (attempt - 1)))


def main():
    CACHE.mkdir(parents=True, exist_ok=True)
    years = list(range(1949, 2027))

    pending = [y for y in years if not (CACHE / f"papers_{y}.parquet").exists()]
    skipped = len(years) - len(pending)
    if skipped:
        print(f"{skipped} años ya en caché, {len(pending)} por descargar.")

    if not pending:
        return

    with Socrata(
        domain="analisi.transparenciacatalunya.cat",
        app_token=os.environ['SOCRATA_APP_TOKEN'],
        username=os.environ['SOCRATA_USER_ID'],
        password=os.environ['SOCRATA_USER_PASSWD'],
        timeout=600,
    ) as client, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:

        futures = {ex.submit(fetch_with_retry, client, y): y for y in pending}

        errors = []
        with tqdm(total=len(futures), desc="Descargando papers", unit="año") as pbar:
            for fut in as_completed(futures):
                year = futures[fut]
                try:
                    _, n = fut.result()
                    pbar.set_postfix_str(f"{year}: {n} filas")
                except Exception as e:
                    errors.append((year, e))
                    pbar.set_postfix_str(f"{year}: ERROR")
                pbar.update(1)

    if errors:
        print("\nAños con error:")
        for year, err in errors:
            print(f"  {year}: {type(err).__name__}: {err}")


if __name__ == "__main__":
    main()

66 años ya en caché, 12 por descargar.


Descargando papers: 100%|██████████| 12/12 [1:22:56<00:00, 414.73s/año, 2021: 594481 filas]


# Data loading

In [48]:
import pandas as pd
from tqdm import tqdm
all_papers = pd.DataFrame()
for f in tqdm(list(CACHE.glob("*.parquet"))):
    if (all_papers.empty):
        all_papers = pd.read_parquet(f)
    else:
        all_papers = pd.concat([all_papers, pd.read_parquet(f)], ignore_index=True)

100%|██████████| 80/80 [00:03<00:00, 22.18it/s]


In [49]:
len(all_papers)

8951767

In [50]:
cleaned_papers = all_papers.drop_duplicates(ignore_index=True)
cleaned_papers.to_parquet(DATA / 'papers_cleaned.gencat.parquet', index=False)
del all_papers 
len(cleaned_papers)

2141817

In [54]:
cleaned_papers = pd.read_parquet(DATA/'papers_cleaned.gencat.parquet')

In [60]:
(
    cleaned_papers['ruct']
    .str.split(',')
    .explode()
    .loc[lambda s: s != '']   # quita cadenas vacías
    .drop_duplicates()
    .dropna()
    .reset_index(drop=True)
).to_frame().to_csv(CACHE / 'ructs.csv', index=False)

In [44]:
directors_mask = cleaned_papers['orcidDirectors'].notna() & cleaned_papers['orcidDirectors'].notnull()

checked_orcids = (
    cleaned_papers['orcidResearchers']
        .str.cat(
            cleaned_papers.loc[directors_mask, 'orcidDirectors'],
            sep=';',
            na_rep=''          # evita NaN al concatenar
        )
        .str.split(';')
        .explode()
        .loc[lambda s: s != '']   # quita cadenas vacías
        .drop_duplicates()
        .reset_index(drop=True)
)

checked_orcids.to_frame().to_parquet(CACHE / 'checked_orcids.parquet', index=False)

In [43]:
# orcidUncheckedAuthors
unckecked_mask = cleaned_papers['orcidUncheckedAuthors'].notna() & cleaned_papers['orcidUncheckedAuthors'].notnull()

unchecked_orcids = (
    cleaned_papers.loc[unckecked_mask, 'orcidUncheckedAuthors']
        .str.split(';')
        .explode()
        .loc[lambda s: s != '']   # quita cadenas vacías
        .drop_duplicates()
        .reset_index(drop=True)
)

unchecked_orcids.to_frame().to_parquet(CACHE / 'unchecked_orcids.parquet', index=False)

In [46]:
cleaned_papers['doi'].drop_duplicates().dropna().to_frame().to_parquet(CACHE/'dois.parquet', index=False)

## Download (all rows) and cache

In [ ]:
def cached(name, fetch):
    """Return data/cache/<name>.parquet; download it with fetch() the first time."""
    path = CACHE / f"{name}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    df = fetch()
    df.to_parquet(path, index=False)
    return df

# df = cached("raw", lambda: fetch_socrata(DATASET))   # fetch_socrata: see 03_open_data_apis
# print(len(df), "rows")

In [47]:
import pandas as pd
pd.read_parquet(CACHE/'dois.parquet')

,doi
0,10.1016/0022-03968790133-1
1,10.1016/0022-510x(87)90095-5
2,10.1016/s0163-10478790892-2
3,10.1016/0197-01868790035-0
4,10.1016/s0277-53870080875-x
...,...
358034,10.1016/0022-328x(93)83411-n
358035,10.1016/0003-2670(93)85329-i
358036,10.3989/pirineos.1993.v141-142.164
358037,10.3989/collectbot.1993.v22.83


In [26]:
from pyorcid import OrcidAuthentication, Orcid

auth = OrcidAuthentication(client_id=os.environ['ORCID_USER_ID'], client_secret=os.environ['ORCID_USER_PASSWD'])
access_token = auth.get_public_access_token()
orcid = Orcid(orcid_id='0000-0003-3619-5519', orcid_access_token=access_token, state="public")

orcid.employments()[0]

[{'Department': 'Electrical and Electronic Engineering',
  'Role': 'PhD Fellow',
  'start-date': '06/2019',
  'end-date': '',
  'organization': 'Universitat de Girona',
  'organization-address': 'Girona, Catalunya, ES',
  'url': 'https://micelab.udg.edu/'},
 {'Department': 'Electrical Engineering',
  'Role': 'Lab Engineer',
  'start-date': '02/2018',
  'end-date': '',
  'organization': 'CECOS University of IT and Emerging Sciences',
  'organization-address': 'Peshawar, Khyber Pakhtunkhwa, PK',
  'url': None},
 {'Department': 'Research and Development',
  'Role': 'Trainee Engineer',
  'start-date': '10/2017',
  'end-date': '02/2018',
  'organization': 'National Radio and Telecommunication Corporation',
  'organization-address': 'Haripur, Khyber Pakhtunkhwa, PK',
  'url': None},
 {'Department': 'Electronic Engineering',
  'Role': 'Graduate Assistant',
  'start-date': '2015',
  'end-date': '2017',
  'organization': 'Ghulam Ishaq Khan Institute of Engineering Sciences and Technology',
  'o

## Checks

Row count against the server, types, missing values.

In [ ]:
def profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (100 * df.isna().mean()).round(1),
        "n_unique": df.nunique(),
    })


# profile(df)

## Dataset card

| Field | Answer |
|---|---|
| Source (URL) and licence | |
| Rows × columns | |
| What is one row? | |
| Columns we use | |
| Missing values / problems | |
| API access? | |